In [1]:
cd /home/lihaoxing/jupyter

/home/lihaoxing/jupyter


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap

# ============================================================
# 1. Settings
# ============================================================
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'svg.fonttype': 'none',
    'svg.hashsalt': 'go_viz'
})

# ============================================================
# 2. Load all 4 files
# ============================================================
FILES = {
    "IVF_2c": "ivf_2c_go_background.txt",
    "IVF_4c": "ivf_4c_go_background.txt",
    "PA_2c":  "pa_2c_go_background.txt",
    "PA_4c":  "pa_4c_go_background.txt",
}

TITLES = {
    "IVF_2c": "IVF 2-cell",
    "IVF_4c": "IVF 4-cell",
    "PA_2c":  "PA 2-cell",
    "PA_4c":  "PA 4-cell",
}

dfs = {}
for label, f in FILES.items():
    d = pd.read_csv(f, sep="\t")
    d = d[["GO", "Description", "LogP", "Enrichment", "Z-score",
           "#GeneInGOAndHitList", "#GeneInGO", "Hits"]].copy()
    d = d.rename(columns={
        "GO": "GO_ID",
        "#GeneInGOAndHitList": "n_hit",
        "#GeneInGO": "n_go",
        "Z-score": "Zscore",
    })
    for c in ["LogP", "Enrichment", "Zscore", "n_hit", "n_go"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["neg_logP"] = -d["LogP"]
    d = d.dropna(subset=["neg_logP"]).reset_index(drop=True)
    dfs[label] = d

# ============================================================
# 3. Filter
# ============================================================
LOG_P_THR = 1.3
TOP_N     = 15

sig = {}
for label, d in dfs.items():
    sub = d[d["neg_logP"] >= LOG_P_THR].copy()
    sub = sub.sort_values("neg_logP", ascending=False).head(TOP_N)
    sig[label] = sub

# ============================================================
# 4. Wrap labels
# ============================================================
def wrap_label(text, width=60):
    return "\n".join(textwrap.wrap(text, width=width))

# ============================================================
# 5. Plot each condition separately
# ============================================================
DS_COLORS = {
    "IVF_2c": "#4C78A8",
    "IVF_4c": "#72B7B2",
    "PA_2c":  "#E45756",
    "PA_4c":  "#F58518",
}

for label, sub in sig.items():

    if len(sub) == 0:
        continue

    n_terms = len(sub)

    # Dynamic figure size
    fig_h = max(4.5, 0.55 * n_terms + 2.0)
    fig_w = 12.5          # 稍宽，让文字区更舒展

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    plot_df = sub.iloc[::-1].reset_index(drop=True)
    y = np.arange(len(plot_df))

    # Bubble（气泡再小一些，避免占满窄区）
    sizes  = 40 + plot_df["n_hit"] * 22
    colors = plot_df["Enrichment"]

    sc = ax.scatter(
        plot_df["neg_logP"], y,
        s=sizes,
        c=colors,
        cmap="viridis",
        edgecolor="black",
        linewidth=0.6,
        alpha=0.88,
        zorder=3
    )

    # Wrapped labels：更宽，多数一行放得下
    labels_y = [wrap_label(s, width=60) for s in plot_df["Description"]]
    ax.set_yticks(y)
    ax.set_yticklabels(labels_y, fontsize=10, linespacing=1.2)

    ax.set_xlabel("−log$_{10}$(P)", fontsize=11)
    ax.set_title(
        f"{TITLES[label]}   ({n_terms} GO terms, "
        f"−log$_{{10}}$(P) ≥ {LOG_P_THR})",
        loc="left", fontsize=12.5, pad=10
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x", linestyle=":", alpha=0.4, zorder=0)

    ax.tick_params(axis="x", labelsize=9.5)

    # ==== 关键：绘图区只占右侧 25% ====
    plt.subplots_adjust(left=0.75, right=0.94)

    # x 轴右边留一点余量
    x_max = plot_df["neg_logP"].max()
    ax.set_xlim(0, x_max * 1.20)

    # Colorbar
    cbar = plt.colorbar(
        sc, ax=ax,
        fraction=0.05, pad=0.03
    )
    cbar.set_label("Enrichment", fontsize=9.5)
    cbar.ax.tick_params(labelsize=8.5)

    svg_file = f"GO_dotplot_{label}.svg"
    pdf_file = f"GO_dotplot_{label}.pdf"

    plt.savefig(svg_file, bbox_inches="tight")
    plt.savefig(pdf_file, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {svg_file}  /  {pdf_file}")

Saved: GO_dotplot_IVF_2c.svg  /  GO_dotplot_IVF_2c.pdf
Saved: GO_dotplot_IVF_4c.svg  /  GO_dotplot_IVF_4c.pdf
Saved: GO_dotplot_PA_2c.svg  /  GO_dotplot_PA_2c.pdf
Saved: GO_dotplot_PA_4c.svg  /  GO_dotplot_PA_4c.pdf
